# SWaT Dataset — Exploratory Data Analysis

**Project:** Unsupervised Anomaly Detection in Industrial Control Systems  
**Dataset:** SWaT (Secure Water Treatment) — iTrust, SUTD Singapore  
**Author:** Ildebrando  

---

## Objectives
1. Understand the SWaT process and sensor layout
2. Load and inspect both normal and attack datasets
3. Assess data quality (missing values, types, timestamps)
4. Profile sensor distributions and dynamics
5. Visualize attack periods and affected sensors
6. Identify correlations and multivariate structure
7. Document findings to guide preprocessing decisions

---

## SWaT Process Overview

SWaT is a scaled-down water treatment plant with 6 sub-processes:

| Stage | Description | Key sensors |
|-------|-------------|-------------|
| P1 | Raw water storage + chemical dosing | FIT101, LIT101, MV101, P101 |
| P2 | Chemical dosing (NaCl, NaOCl, HCl) | AIT201, AIT202, AIT203 |
| P3 | Ultrafiltration | FIT301, LIT301, DPIT301 |
| P4 | De-chlorination (UV) | AIT401, AIT402 |
| P5 | Reverse osmosis | AIT501–503, FIT501–504 |
| P6 | Permeate backwash | FIT601, P601 |

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_RAW = Path('data/raw')
DATA_PROCESSED = Path('data/processed')
FIGURES = Path('reports/figures')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGSIZE = (16, 5)

print('Setup complete')
print(f'Raw data path: {DATA_RAW.resolve()}')

## 1. Load Data

In [ ]:
# ---- Load normal operation data ----
df_normal = pd.read_excel(
    DATA_RAW / 'SWaT_Dataset_Normal_v1.xlsx',
    header=1          # SWaT xlsx has a merged header row — adjust if needed
)

# ---- Load attack data ----
df_attack = pd.read_excel(
    DATA_RAW / 'SWaT_Dataset_Attack_v0.xlsx',
    header=1
)

print(f'Normal dataset:  {df_normal.shape[0]:,} rows × {df_normal.shape[1]} columns')
print(f'Attack dataset:  {df_attack.shape[0]:,} rows × {df_attack.shape[1]} columns')

In [ ]:
# ---- Quick inspection ----
print('=== NORMAL — first rows ===')
df_normal.head(3)

In [ ]:
print('=== ATTACK — first rows ===')
df_attack.head(3)

In [ ]:
# ---- Column names ----
print('Columns:', df_normal.columns.tolist())

## 2. Data Cleaning & Types

In [ ]:
# ---- Parse timestamp ----
# SWaT timestamp column is typically named ' Timestamp' (note leading space)
TIMESTAMP_COL = ' Timestamp'   # adjust if different
LABEL_COL     = 'Normal/Attack'

for df, name in [(df_normal, 'normal'), (df_attack, 'attack')]:
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
    df.set_index(TIMESTAMP_COL, inplace=True)
    df.sort_index(inplace=True)
    print(f'{name}: {df.index.min()} → {df.index.max()}')

In [ ]:
# ---- Separate features from label ----
# Keep label only in attack df (used ONLY for evaluation, not training)
y_attack = (df_attack[LABEL_COL].str.strip() == 'Attack').astype(int)

# Sensor/actuator columns (all except label)
FEATURE_COLS = [c for c in df_normal.columns if c != LABEL_COL]

X_normal = df_normal[FEATURE_COLS].copy()
X_attack = df_attack[FEATURE_COLS].copy()

print(f'Feature columns: {len(FEATURE_COLS)}')
print(f'Attack label distribution:\n{y_attack.value_counts()}')

In [ ]:
# ---- Data types audit ----
print('=== Normal dataset dtypes ===')
print(X_normal.dtypes.value_counts())
print()

# Identify actuator columns (binary 0/1 or categorical) vs continuous sensors
binary_cols = [c for c in FEATURE_COLS if X_normal[c].nunique() <= 2]
continuous_cols = [c for c in FEATURE_COLS if c not in binary_cols]

print(f'Continuous sensors:  {len(continuous_cols)}')
print(f'Binary actuators:    {len(binary_cols)}')
print(f'Binary cols: {binary_cols}')

In [ ]:
# ---- Missing values ----
missing_normal = X_normal.isnull().sum()
missing_attack = X_attack.isnull().sum()

print('Missing values — normal:', missing_normal.sum())
print('Missing values — attack:', missing_attack.sum())

# If any, inspect which columns
if missing_normal.sum() > 0:
    print(missing_normal[missing_normal > 0])

In [ ]:
# ---- Sampling frequency check ----
time_diffs = pd.Series(X_normal.index).diff().dropna()
print('Sampling interval:')
print(time_diffs.describe())
# Expected: 1 second

## 3. Statistical Summary

In [ ]:
# ---- Descriptive stats — continuous sensors ----
stats = X_normal[continuous_cols].describe().T
stats['cv'] = stats['std'] / stats['mean'].abs()  # coefficient of variation
stats.sort_values('cv', ascending=False).round(3)

In [ ]:
# ---- Distribution plots — continuous sensors ----
n_cols = 4
n_rows = int(np.ceil(len(continuous_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    axes[i].hist(X_normal[col].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='none')
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sensor Distributions — Normal Operation', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'sensor_distributions_normal.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Time-Series Visualization

In [ ]:
# ---- Full time-series — key sensors per sub-process ----
# Representative sensor per stage — adjust after inspecting column names
KEY_SENSORS = {
    'P1 - Raw water (LIT101)': 'LIT101',
    'P2 - Conductivity (AIT201)': 'AIT201',
    'P3 - UF feed flow (FIT301)': 'FIT301',
    'P4 - Cl residual (AIT402)': 'AIT402',
    'P5 - RO feed flow (FIT501)': 'FIT501',
}

fig, axes = plt.subplots(len(KEY_SENSORS), 1, figsize=(18, 12), sharex=True)

for ax, (label, sensor) in zip(axes, KEY_SENSORS.items()):
    if sensor in X_normal.columns:
        ax.plot(X_normal.index, X_normal[sensor], lw=0.4, color='steelblue')
        ax.set_ylabel(label, fontsize=8)
        ax.tick_params(labelsize=7)
    else:
        ax.set_visible(False)
        print(f'Column not found: {sensor} — update KEY_SENSORS dict')

plt.suptitle('Key Sensors — Normal Operation (7 days)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'timeseries_normal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Attack dataset: overlay attack periods ----
fig, axes = plt.subplots(len(KEY_SENSORS), 1, figsize=(18, 12), sharex=True)

for ax, (label, sensor) in zip(axes, KEY_SENSORS.items()):
    if sensor not in X_attack.columns:
        continue
    ax.plot(X_attack.index, X_attack[sensor], lw=0.4, color='steelblue', label='Normal')
    # Shade attack windows
    attack_mask = y_attack.values.astype(bool)
    attack_times = X_attack.index[attack_mask]
    if len(attack_times) > 0:
        # Group contiguous attack periods
        breaks = np.where(np.diff(attack_mask.astype(int)) != 0)[0] + 1
        periods = np.split(np.where(attack_mask)[0], breaks[breaks < len(attack_mask)])
        for period in periods:
            if len(period) > 0:
                ax.axvspan(X_attack.index[period[0]], X_attack.index[period[-1]],
                           alpha=0.25, color='red', label='_nolegend_')
    ax.set_ylabel(label, fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].legend(['Normal', 'Attack window'], fontsize=8)
plt.suptitle('Key Sensors — Attack Dataset (red = attack period)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'timeseries_attacks.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Attack Summary

In [ ]:
# ---- Attack period statistics ----
attack_pct = y_attack.mean() * 100
print(f'Attack rows:  {y_attack.sum():,} ({attack_pct:.1f}%)')
print(f'Normal rows:  {(y_attack == 0).sum():,} ({100 - attack_pct:.1f}%)')

# Count distinct attack episodes
attack_changes = y_attack.diff().fillna(0)
n_episodes = (attack_changes == 1).sum()
print(f'\nDistinct attack episodes: {n_episodes}')

In [ ]:
# ---- Per-sensor: mean shift during attacks vs normal ----
normal_means = X_attack[continuous_cols][y_attack == 0].mean()
attack_means = X_attack[continuous_cols][y_attack == 1].mean()

mean_shift = ((attack_means - normal_means) / (normal_means.abs() + 1e-6) * 100)
mean_shift_sorted = mean_shift.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
mean_shift_sorted.head(20).plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Top 20 Sensors by Mean Shift During Attacks (%)', fontsize=11)
ax.set_ylabel('|% change in mean|')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(FIGURES / 'sensor_mean_shift_attacks.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
# ---- Correlation heatmap — normal operation ----
corr = X_normal[continuous_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False, cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax
)
ax.set_title('Sensor Correlation Matrix — Normal Operation', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'correlation_matrix_normal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Highly correlated pairs ----
corr_upper = corr.where(mask == False)
high_corr = (
    corr_upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'sensor_1', 'level_1': 'sensor_2', 0: 'correlation'})
)
high_corr['abs_corr'] = high_corr['correlation'].abs()
high_corr = high_corr[high_corr['abs_corr'] > 0.85].sort_values('abs_corr', ascending=False)

print(f'Highly correlated pairs (|r| > 0.85): {len(high_corr)}')
high_corr.head(15)

## 7. Stationarity & Variability Check

In [ ]:
# ---- Rolling mean/std to spot non-stationarity ----
# Resample to 1-min to reduce noise
X_normal_1min = X_normal[continuous_cols].resample('1min').mean()

WINDOW = 60  # 60-min rolling window

sensor_to_check = continuous_cols[0]  # replace with sensor of interest

fig, axes = plt.subplots(2, 1, figsize=FIGSIZE, sharex=True)
axes[0].plot(X_normal_1min[sensor_to_check], lw=0.8, label='1-min mean')
axes[0].plot(X_normal_1min[sensor_to_check].rolling(WINDOW).mean(), lw=1.5, color='red', label=f'{WINDOW}-min rolling mean')
axes[0].set_title(f'{sensor_to_check} — Rolling Mean')
axes[0].legend(fontsize=8)

axes[1].plot(X_normal_1min[sensor_to_check].rolling(WINDOW).std(), lw=0.8, color='orange')
axes[1].set_title(f'{sensor_to_check} — Rolling Std')

plt.tight_layout()
plt.show()

In [ ]:
# ---- Coefficient of variation per sensor ----
cv = (X_normal[continuous_cols].std() / X_normal[continuous_cols].mean().abs()).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
cv.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Coefficient of Variation — Continuous Sensors (Normal Operation)', fontsize=11)
ax.set_ylabel('CV = std / |mean|')
ax.tick_params(axis='x', rotation=45, labelsize=7)
plt.tight_layout()
plt.savefig(FIGURES / 'coefficient_of_variation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. EDA Summary & Preprocessing Notes

In [ ]:
# ---- Summary to carry into notebook 02 ----
summary = {
    'n_features_total': len(FEATURE_COLS),
    'n_continuous': len(continuous_cols),
    'n_binary_actuators': len(binary_cols),
    'n_normal_rows': len(X_normal),
    'n_attack_rows': len(X_attack),
    'attack_pct': round(y_attack.mean() * 100, 2),
    'n_attack_episodes': int(n_episodes),
    'missing_values': bool(X_normal.isnull().sum().sum() > 0),
    'sampling_freq_seconds': 1,
    'high_corr_pairs': len(high_corr),
}

print('=== EDA Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

print()
print('=== Preprocessing checklist for notebook 02 ===')
print('  [ ] Parse and align timestamps')
print('  [ ] Handle missing values (forward fill or interpolate)')
print('  [ ] Separate continuous vs binary features')
print('  [ ] StandardScaler on continuous features (fit on NORMAL only)')
print('  [ ] Consider resampling (1-min avg) to reduce memory + noise')
print('  [ ] Engineer rolling features: mean_10s, std_10s, rate_of_change')
print('  [ ] Save processed parquet files for modeling notebooks')
print('  [ ] Keep y_attack separate — do NOT use for training')

---
**Next:** `02_preprocessing.ipynb` — feature engineering, scaling, train/test split